# PyTorch & Deep Learning for QR Interviews
*A practical guide focused on what Citadel, HRT, Jump, etc. actually ask.*

---

## Why this notebook exists
QR interviews don't ask you to call `nn.Linear`. They ask you to **implement it**, including the backward pass, and write test cases. From real interview logs, the questions were things like:

- *"Use the `torch` package to implement the Leaky-ReLU activation function from scratch. Must write both the forward and backward (gradient) propagation. Must write your own test cases."* — **HRT**
- *"Write pseudo-code for the Dropout layer (specifically for the feedforward pass)."* — **Citadel GQS**
- *"Replicate `pandas.DataFrame.agg`."* — **HRT** (same spirit: rebuild a primitive, don't use it)

Behind all of these is the same question: **do you understand the primitive well enough to rebuild it, and do you trust your code enough to write tests?**

This notebook walks through that core skill. By the end you should be able to:

1. Reason about the autograd graph and the chain rule **in code**
2. Implement any layer two ways — composing differentiable ops (`nn.Module`) and writing the backward by hand (`torch.autograd.Function`)
3. Verify your implementation with `gradcheck` and against PyTorch's built-ins
4. Run a full training loop on tabular data using *your own* layers

## How to use it
Each section is short theory + code small enough to write in a 30-min interview. The Dropout and LeakyReLU sections are the actual interview questions worked end-to-end — treat them as templates.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import gradcheck
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
print(f"torch {torch.__version__}")

torch 2.12.0+cu130


## 1. Tensors and autograd — the 60-second refresher

A PyTorch tensor is a NumPy array plus optional gradient tracking. When `requires_grad=True`, PyTorch builds a **computational graph** of every op you do, so calling `.backward()` later replays the chain rule.

Three things to internalize:

1. **Leaves vs non-leaves.** A tensor you create with `requires_grad=True` (or an `nn.Parameter`) is a *leaf*; its `.grad` accumulates after backward. Tensors produced by ops are non-leaf; their gradients are computed in passing but not stored by default.
2. **`grad_fn`** is the operation that produced a non-leaf tensor. Backward walks from the scalar loss back through `grad_fn` pointers, applying each op's local Jacobian.
3. **`.backward()` *accumulates*.** Always zero gradients (`opt.zero_grad()` or `x.grad = None`) before the next backward, or you'll silently sum gradients across iterations. This is one of the top interview "gotcha" questions.


In [2]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
y.backward()

print("d(sum(x^2))/dx =", x.grad)        # tensor([2., 4., 6.])
print("y.grad_fn =", y.grad_fn)            # SumBackward0
print("x.grad_fn =", x.grad_fn)            # None — x is a leaf
print("x.is_leaf =", x.is_leaf)            # True

d(sum(x^2))/dx = tensor([2., 4., 6.])
y.grad_fn = <SumBackward0 object at 0x7f8aa8eb3730>
x.grad_fn = None
x.is_leaf = True


## 2. Two ways to write a custom layer

| Method | Backward you write | When to use |
|---|---|---|
| `nn.Module` composing existing ops | None — autograd does it | When the layer is a composition of differentiable ops |
| `torch.autograd.Function` | Yes, by hand | When the op is novel, when you want a more efficient/stable backward, or when the interviewer asks |

**Interview rule of thumb:** if the question is "implement X *from scratch*" or "write the backward by hand", use `torch.autograd.Function`. If they just say "build a layer", `nn.Module` is fine.

### Anatomy of `torch.autograd.Function`

```python
class MyFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, *inputs):
        ctx.save_for_backward(...)   # save tensors needed by backward
        ctx.some_scalar = ...         # save non-tensor scalars as attributes
        return output

    @staticmethod
    def backward(ctx, grad_output):
        saved = ctx.saved_tensors
        # return one grad per forward input; None for non-tensor inputs
        return grad_wrt_input1, grad_wrt_input2, ...
```

Call it via `MyFn.apply(...)`, **not** `MyFn(...)`. That's the most common mistake.


## 3. Custom Dropout
*(Citadel GQS: "Write pseudo-code for the Dropout layer.")*

### Theory
During training, set each unit to zero independently with probability $p$. To keep the *expected* output equal to the input, scale survivors by $1/(1-p)$ — this is **inverted dropout**. At eval time, dropout is the identity.

Why scale? Let $z = x \cdot m / (1-p)$ where $m \sim \text{Bernoulli}(1-p)$. Then $\mathbb{E}[z] = x \cdot \mathbb{E}[m] / (1-p) = x$. So the next layer sees the same expected activations at train and eval time, *without* needing different weights at eval — eval is just identity.

### Backward
The op is just elementwise multiply by the mask (which is constant w.r.t. $x$), so
$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} \cdot \frac{m}{1-p}$$
We don't even need a custom `Function` — composing ops in `nn.Module` lets autograd handle the backward automatically.


In [3]:
class MyDropout(nn.Module):
    """Inverted dropout. Matches torch.nn.Dropout semantics."""
    def __init__(self, p: float = 0.5):
        super().__init__()
        assert 0.0 <= p < 1.0, "p must be in [0, 1)"
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        # mask = 1 with prob (1-p), 0 with prob p
        mask = (torch.rand_like(x) > self.p).to(x.dtype)
        return x * mask / (1.0 - self.p)


# Quick sanity check vs nn.Dropout
torch.manual_seed(42)
x = torch.randn(10_000)
mine = MyDropout(0.3)

mine.train()
out = mine(x)
print(f"mean(x)          = {x.mean().item():+.4f}")
print(f"mean(dropout(x)) = {out.mean().item():+.4f}    (should match)")
print(f"fraction zeroed  = {(out == 0).float().mean().item():.4f}    (~0.30 expected)")

mine.eval()
assert torch.allclose(mine(x), x), "Eval mode must be identity"
print("eval mode: identity ✓")

mean(x)          = +0.0064
mean(dropout(x)) = +0.0018    (should match)
fraction zeroed  = 0.3112    (~0.30 expected)
eval mode: identity ✓


### Common interview follow-ups on Dropout
- **"What if you don't scale by $1/(1-p)$?"** — Then activations at train time are systematically smaller than at eval, so weights would need to be rescaled at eval time ("standard dropout"). Inverted dropout is preferred because eval is free.
- **"How does it differ from DropConnect?"** — DropConnect zeros *weights*, not activations.
- **"Why does dropout regularize?"** — Roughly: training with dropout is approximating an ensemble of $2^n$ sub-networks, and the noise reduces co-adaptation between units. (Ref: Goodfellow §7.12.)
- **"Where does PyTorch check the mode?"** — `self.training`. Inherit from `nn.Module` so `model.train()` / `model.eval()` flip it for the whole tree.
- **"Can you write it as an `autograd.Function`?"** — Yes, but you'd just be re-implementing what elementwise multiply already does. Only worth it if you needed to fuse it with another op.


## 4. Custom LeakyReLU — the canonical HRT question

*"Use the `torch` package to implement Leaky-ReLU from scratch. Write both forward and backward, plus your own test cases."*

### Theory
$$
f(x) = \begin{cases} x & x > 0 \\ \alpha x & x \le 0 \end{cases}, \qquad
f'(x) = \begin{cases} 1 & x > 0 \\ \alpha & x \le 0 \end{cases}
$$

where $\alpha$ ("negative_slope") is typically 0.01. The point of LeakyReLU vs ReLU is that the gradient on the negative side is nonzero, mitigating the "dying ReLU" problem.

We'll implement it **two ways**:
- **(a)** as an `nn.Module` composing built-in ops (autograd handles backward),
- **(b)** as a `torch.autograd.Function` where we write the backward by hand.

The interview wants (b). Always show both — it demonstrates you know *when* to write a custom Function and when not to.


In [4]:
# (a) nn.Module — autograd computes the gradient automatically
class LeakyReLUModule(nn.Module):
    def __init__(self, negative_slope: float = 0.01):
        super().__init__()
        self.negative_slope = negative_slope

    def forward(self, x):
        return torch.where(x > 0, x, self.negative_slope * x)

In [5]:
# (b) torch.autograd.Function — manual backward
class LeakyReLUFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, negative_slope):
        ctx.save_for_backward(x)
        ctx.negative_slope = negative_slope         # non-tensor → save as attr
        return torch.where(x > 0, x, negative_slope * x)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        # local derivative: 1 where x > 0, else negative_slope
        local = torch.where(
            x > 0,
            torch.ones_like(x),
            torch.full_like(x, ctx.negative_slope),
        )
        # chain rule: dL/dx = dL/dy * dy/dx
        grad_x = grad_output * local
        return grad_x, None       # None grad for the non-tensor arg


# Convenience wrapper so callers don't have to remember .apply
def my_leaky_relu(x, negative_slope: float = 0.01):
    return LeakyReLUFn.apply(x, negative_slope)

### Test cases
The interviewer explicitly asks for these. A good test suite for any custom op covers:

1. **Forward matches the reference implementation** on a random input.
2. **Backward matches** the reference's gradient on the same input.
3. **Edge cases**: zero input, all-positive input, all-negative input, very large input.
4. **`gradcheck`**: finite-difference verification of your backward.


In [6]:
def test_leaky_relu(slope=0.1, shape=(7, 11), dtype=torch.float64):
    # --- forward + backward vs PyTorch's reference ----------------------
    x = torch.randn(*shape, dtype=dtype, requires_grad=True)
    x_ref = x.detach().clone().requires_grad_(True)

    out_mine = my_leaky_relu(x, slope)
    out_ref  = F.leaky_relu(x_ref, slope)
    assert torch.allclose(out_mine, out_ref), "forward mismatch"

    g = torch.randn_like(out_mine)
    out_mine.backward(g)
    out_ref.backward(g)
    assert torch.allclose(x.grad, x_ref.grad), "backward mismatch"

    # --- edge cases ------------------------------------------------------
    # x = 0: by PyTorch convention, derivative uses the negative-slope branch
    z = torch.zeros(1, dtype=dtype, requires_grad=True)
    out = my_leaky_relu(z, slope)
    out.backward()
    assert torch.allclose(z.grad, torch.tensor([slope], dtype=dtype))

    # all-positive: gradient should be all ones
    p = torch.full((5,), 3.0, dtype=dtype, requires_grad=True)
    my_leaky_relu(p, slope).sum().backward()
    assert torch.allclose(p.grad, torch.ones(5, dtype=dtype))

    # all-negative: gradient should be all slope
    n = torch.full((5,), -2.0, dtype=dtype, requires_grad=True)
    my_leaky_relu(n, slope).sum().backward()
    assert torch.allclose(n.grad, torch.full((5,), slope, dtype=dtype))

    print("forward, backward, x=0, all-pos, all-neg ✓")

test_leaky_relu()

forward, backward, x=0, all-pos, all-neg ✓


### `gradcheck` — the gold-standard verification

`torch.autograd.gradcheck` does **finite-difference numerical gradients** and compares them to your analytical backward. If they match within tolerance, your backward is correct.

Two gotchas:
- Use `dtype=torch.float64`. Finite differences are noisy in `float32`.
- The function passed to `gradcheck` must take only tensor inputs that require grad — wrap non-tensor args in a closure.


In [7]:
x = torch.randn(4, 5, dtype=torch.float64, requires_grad=True)
ok = gradcheck(lambda inp: LeakyReLUFn.apply(inp, 0.1), (x,), eps=1e-6, atol=1e-5)
print(f"gradcheck passed: {ok} ✓")

gradcheck passed: True ✓


## 5. Three more layers in the same pattern

These are common warmups before the LeakyReLU question. Same recipe each time: write `forward`, derive $\partial \text{out}/\partial \text{in}$ on paper, code the backward, gradcheck.

### ReLU
$$f(x) = \max(0, x), \qquad f'(x) = \mathbf{1}[x > 0]$$

### Sigmoid
$$\sigma(x) = \frac{1}{1+e^{-x}}, \qquad \sigma'(x) = \sigma(x)(1 - \sigma(x))$$
**Trick:** save the *output*, not the input — the derivative is in terms of $\sigma(x)$ itself, so saving the output skips one redundant computation.

### Linear (matmul + bias)
$$y = x W^\top + b$$
For batched $x \in \mathbb{R}^{N \times D_\text{in}}$, $W \in \mathbb{R}^{D_\text{out} \times D_\text{in}}$, $b \in \mathbb{R}^{D_\text{out}}$:
- $\partial L/\partial x = \partial L/\partial y \cdot W$
- $\partial L/\partial W = (\partial L/\partial y)^\top x$
- $\partial L/\partial b = \sum_i \partial L/\partial y_i$ (sum over batch)

The bias gradient summing over the batch is the classic mistake — easy to write `grad_output` instead of `grad_output.sum(0)`.


In [8]:
class ReLUFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp(min=0)
    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * (x > 0).to(grad_output.dtype)


class SigmoidFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        y = torch.sigmoid(x)
        ctx.save_for_backward(y)              # save OUTPUT, not input
        return y
    @staticmethod
    def backward(ctx, grad_output):
        y, = ctx.saved_tensors
        return grad_output * y * (1 - y)


class LinearFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b):
        ctx.save_for_backward(x, W)
        return x @ W.t() + b
    @staticmethod
    def backward(ctx, grad_output):
        x, W = ctx.saved_tensors
        grad_x = grad_output @ W
        grad_W = grad_output.t() @ x
        grad_b = grad_output.sum(dim=0)
        return grad_x, grad_W, grad_b


# gradcheck all three ---------------------------------------------------------
x = torch.randn(3, 4, dtype=torch.float64, requires_grad=True)
assert gradcheck(ReLUFn.apply,    (x,))
assert gradcheck(SigmoidFn.apply, (x,))

W = torch.randn(5, 4, dtype=torch.float64, requires_grad=True)
b = torch.randn(5,    dtype=torch.float64, requires_grad=True)
assert gradcheck(LinearFn.apply, (x, W, b))

print("ReLU, Sigmoid, Linear: gradcheck all passed ✓")

ReLU, Sigmoid, Linear: gradcheck all passed ✓


## 6. Softmax + Cross-Entropy: why frameworks combine them

A classic interview question: *"Why do frameworks expose `cross_entropy(logits, target)` instead of `nll_loss(log(softmax(logits)), target)`?"*

Two reasons:

**(1) Numerical stability.** Softmax involves `exp`, which overflows for large logits. The standard trick is to subtract the max before exponentiating:
$$\log \text{softmax}(z)_i = z_i - \max_j z_j - \log \sum_j \exp(z_j - \max_j z_j)$$

**(2) The combined gradient is dramatically simpler.** With $p = \text{softmax}(z)$ and one-hot label $y$:
$$\frac{\partial L}{\partial z_i} = p_i - y_i$$

No cancellation, no large/small ratios. This is the canonical example of "writing your own backward is sometimes *simpler* than what autograd would chain together."


In [9]:
class SoftmaxCEFn(torch.autograd.Function):
    """Numerically stable softmax + cross-entropy (mean reduction).
    Input: logits (N, C); target: class index (N,)."""
    @staticmethod
    def forward(ctx, logits, target):
        m = logits.max(dim=1, keepdim=True).values
        log_sum_exp = (logits - m).exp().sum(dim=1, keepdim=True).log() + m
        log_p = logits - log_sum_exp                                  # (N, C)
        nll = -log_p.gather(1, target.unsqueeze(1)).squeeze(1)        # (N,)
        ctx.save_for_backward(log_p.exp(), target)
        ctx.N = logits.size(0)
        return nll.mean()

    @staticmethod
    def backward(ctx, grad_output):
        p, target = ctx.saved_tensors
        N = ctx.N
        grad = p.clone()
        grad[torch.arange(N), target] -= 1.0          # (p - y)
        return grad * (grad_output / N), None         # divide by N for mean


# Verify against torch.nn.functional.cross_entropy
torch.manual_seed(0)
logits = torch.randn(8, 5, requires_grad=True)
target = torch.randint(0, 5, (8,))

loss_mine = SoftmaxCEFn.apply(logits, target)
loss_mine.backward()
g_mine = logits.grad.clone()
logits.grad = None

loss_ref = F.cross_entropy(logits, target)
loss_ref.backward()
g_ref = logits.grad

assert torch.allclose(loss_mine, loss_ref, atol=1e-6)
assert torch.allclose(g_mine,    g_ref,    atol=1e-6)
print(f"loss  : {loss_mine.item():.6f}  vs  {loss_ref.item():.6f}  ✓")
print(f"max |grad diff| = {(g_mine - g_ref).abs().max().item():.2e}  ✓")

loss  : 1.891258  vs  1.891258  ✓
max |grad diff| = 7.45e-09  ✓


## 7. End-to-end: train a tabular MLP using *your* layers

Most QR data-science rounds (HRT, Jump) give you a tabular dataset and ask you to fit a model. Here we wire together our custom `LeakyReLU` and `Dropout` into a 3-layer MLP and run a full training loop on synthetic data shaped like such a task.

The point: in the interview you should be able to write this loop from blank screen in ~5 minutes.


In [10]:
# Synthetic regression with mild nonlinearity and noise
torch.manual_seed(0)
N, D = 2000, 12
X = torch.randn(N, D)
true_w  = torch.randn(D)
true_nl = torch.randn(D)
y = X @ true_w + 0.3 * (X ** 2) @ true_nl + 0.5 * torch.randn(N)

# Time-style split — never shuffle for finance data
ntrain = 1600
X_tr, X_va = X[:ntrain], X[ntrain:]
y_tr, y_va = y[:ntrain], y[ntrain:]


class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden=64, p_drop=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 1)
        self.drop = MyDropout(p_drop)            # our custom dropout

    def forward(self, x):
        h = my_leaky_relu(self.fc1(x), 0.01)     # our custom leaky relu
        h = self.drop(h)
        h = my_leaky_relu(self.fc2(h), 0.01)
        h = self.drop(h)
        return self.fc3(h).squeeze(-1)


model = TabularMLP(D)
opt   = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(80):
    model.train()
    opt.zero_grad()
    loss = F.mse_loss(model(X_tr), y_tr)
    loss.backward()
    opt.step()

    if epoch % 10 == 0 or epoch == 79:
        model.eval()
        with torch.no_grad():
            val = F.mse_loss(model(X_va), y_va).item()
        print(f"epoch {epoch:3d}  train MSE {loss.item():.4f}   val MSE {val:.4f}")

epoch   0  train MSE 16.3566   val MSE 14.1617
epoch  10  train MSE 4.0969   val MSE 3.1693
epoch  20  train MSE 2.6311   val MSE 1.6567
epoch  30  train MSE 1.7616   val MSE 0.9479


epoch  40  train MSE 1.3173   val MSE 0.6810
epoch  50  train MSE 1.1924   val MSE 0.5355
epoch  60  train MSE 1.0443   val MSE 0.4656
epoch  70  train MSE 0.9366   val MSE 0.4330


epoch  79  train MSE 0.9125   val MSE 0.4242


## 8. The `train()` / `eval()` gotcha

Forgetting `model.eval()` at inference is one of the most common production bugs and a top interview gotcha. Dropout, BatchNorm, and a few others change behavior between modes.


In [11]:
torch.manual_seed(0)
d = MyDropout(0.5)
x = torch.ones(8)

d.train()
print("train mode (random each call, scaled by 1/(1-p)=2):")
for _ in range(3):
    print(" ", d(x))

d.eval()
print("\neval mode (identity, deterministic):")
print(" ", d(x))

train mode (random each call, scaled by 1/(1-p)=2):
  tensor([0., 2., 0., 0., 0., 2., 0., 2.])
  tensor([0., 2., 0., 0., 0., 0., 0., 2.])
  tensor([2., 2., 0., 0., 2., 2., 0., 2.])

eval mode (identity, deterministic):
  tensor([1., 1., 1., 1., 1., 1., 1., 1.])


## 9. Pandas appendix — the data-prep that comes *before* the model

QR rounds (HRT insurance task, Jump pets+claims) frame the task as *"given this DataFrame, predict the target."* The model is 30% of the work — preparing the data is the other 70%. A compact reference for the patterns mentioned in real interview logs:

| Task | Idiom |
|---|---|
| Impute numeric NaN | `df[c].fillna(df[c].median())` |
| Impute categorical NaN | `df[c].fillna(df[c].mode()[0])` or sentinel `"missing"` |
| One-hot encode | `pd.get_dummies(df, columns=cat_cols, drop_first=True)` |
| Standardize | `from sklearn.preprocessing import StandardScaler` |
| Heavy-tailed target | `np.log1p(y)` — avoids `log(0)`; classic for claims, income, counts |
| Outlier clipping | `df[c].clip(df[c].quantile(0.01), df[c].quantile(0.99))` |
| Conditional features | `np.where(cond, a, b)` |
| Join tables | `pd.merge(left, right, on='key', how='left')` |
| High-cardinality cats | Frequency or target encoding, **not** one-hot |
| Time-aware split | Always split by time for finance — never shuffle |

### What to say *out loud* in the interview

- **"Why a linear model?"** → "Better interpretability, easier to diagnose, and a strong baseline before reaching for trees or NNs."
- **"What metric?"** → "If the target is heavy-tailed (claims, returns), MSE rewards outliers too much. Prefer **MAE**, **Huber**, or model `log1p(y)` and report metrics in log-space."
- **"What model first?"** → "For small tabular data, gradient-boosted trees (`XGBoost`, `LightGBM`) usually dominate. Neural nets pay off when $N$ is large and features have structure (sequences, images, text). I'd start with a tree."
- **"How do you handle high-cardinality categoricals?"** → "Frequency-encode or target-encode (with out-of-fold). One-hot blows up dimensionality and is statistically inefficient."

### Mini demo of the idioms


In [12]:
import pandas as pd

rng = np.random.default_rng(0)
df = pd.DataFrame({
    'age':    rng.integers(18, 80, 100).astype(float),
    'income': rng.lognormal(10, 1, 100),
    'region': rng.choice(['NY', 'SF', 'LA', None], 100),
    'claims': rng.poisson(2, 100),
})
df.loc[rng.choice(100, 10, replace=False), 'age'] = np.nan

# Impute -----------------------------------------------------------------
df['age']    = df['age'].fillna(df['age'].median())
df['region'] = df['region'].fillna('unknown')

# Heavy-tailed feature ---------------------------------------------------
df['log_income'] = np.log1p(df['income'])

# Outlier clip (winsorize) ----------------------------------------------
lo, hi = df['income'].quantile([0.01, 0.99])
df['income'] = df['income'].clip(lo, hi)

# Conditional feature ---------------------------------------------------
df['is_senior'] = np.where(df['age'] >= 65, 1, 0)

# One-hot --------------------------------------------------------------
df_enc = pd.get_dummies(df, columns=['region'], drop_first=True)

print(df_enc.head())
print(f"\nshape after encoding: {df_enc.shape}")
print(f"remaining NaNs: {int(df_enc.isna().sum().sum())}")

    age        income  claims  log_income  is_senior  region_NY  region_SF  \
0  70.0  31488.586023       1   10.357412          1      False       True   
1  57.0   6579.285078       2    8.791833          0      False      False   
2  49.0  21928.575155       2    9.995591          0      False      False   
3  34.0  42466.675229       1   10.656498          0      False       True   
4  37.0   6073.185474       1    8.711803          0      False      False   

   region_unknown  
0           False  
1           False  
2           False  
3           False  
4            True  

shape after encoding: (100, 8)
remaining NaNs: 0


In [13]:
# Quick EDA template — the 5 lines you should always run on a new dataset
print("--- info ---")
df_enc.info()
print("\n--- describe (numeric) ---")
print(df_enc.describe().round(2))
print("\n--- nulls ---")
print(df_enc.isna().sum())
print("\n--- skew (heavy-tailed features?) ---")
print(df_enc.select_dtypes(include=np.number).skew().sort_values(ascending=False).round(2))

--- info ---
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             100 non-null    float64
 1   income          100 non-null    float64
 2   claims          100 non-null    int64  
 3   log_income      100 non-null    float64
 4   is_senior       100 non-null    int64  
 5   region_NY       100 non-null    bool   
 6   region_SF       100 non-null    bool   
 7   region_unknown  100 non-null    bool   
dtypes: bool(3), float64(3), int64(2)
memory usage: 4.3 KB

--- describe (numeric) ---


          age     income  claims  log_income  is_senior
count  100.00     100.00  100.00      100.00     100.00
mean    48.63   34521.12    2.21       10.03       0.19
std     16.75   31364.41    1.73        0.98       0.39
min     18.00    2318.72    0.00        7.64       0.00
25%     37.75   11222.78    1.00        9.33       0.00
50%     50.00   26235.24    2.00       10.17       0.00
75%     60.50   44517.63    3.00       10.70       0.00
max     79.00  136487.20    9.00       12.00       1.00

--- nulls ---
age               0
income            0
claims            0
log_income        0
is_senior         0
region_NY         0
region_SF         0
region_unknown    0
dtype: int64

--- skew (heavy-tailed features?) ---
is_senior     1.60
income        1.50
claims        1.38
age          -0.24
log_income   -0.25
dtype: float64


## 10. Resources to drill further

### Books / chapters most relevant to the questions in your log

- **Goodfellow, Bengio, Courville — *Deep Learning*** (free at [deeplearningbook.org](https://www.deeplearningbook.org)).
  - Ch. 6 (Feedforward Networks) — chain rule + backprop in clean notation.
  - Ch. 7 (Regularization) — **§7.12 is *the* reference on dropout / inverted dropout**.
  - Ch. 8 (Optimization) — SGD, momentum, Adam derivations.

- **Bishop — *Pattern Recognition and Machine Learning*.**
  - Ch. 5 (Neural Networks) — backprop done the clean way.
  - Ch. 3 (Linear Models) — OLS/Ridge derivations Citadel asks for every year.

- **Hastie, Tibshirani, Friedman — *Elements of Statistical Learning*.**
  - Ch. 3 — Ridge/Lasso constraint-region geometry (Citadel's "visualize L1 vs L2" question).

- **Zhang, Lipton, Li, Smola — *Dive into Deep Learning* ([d2l.ai](https://d2l.ai)).** Working PyTorch code for everything. The "from scratch" sections — especially [Dropout from scratch](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html) — are exactly the interview style.

### Highly recommended for autograd intuition

- **Andrej Karpathy — *Neural Networks: Zero to Hero* (YouTube).** The micrograd lecture builds autograd from scratch in pure Python. After watching it, `torch.autograd.Function` is obvious.
- **PyTorch docs → "Extending PyTorch with Custom Autograd Functions"** ([link](https://pytorch.org/docs/stable/notes/extending.html)). Short, direct, exactly the LeakyReLU question.

### Drilling routine for the week before an interview

1. Re-implement, blank-screen, in under 15 min each: **ReLU, LeakyReLU, ELU, Sigmoid, Tanh, Dropout, Linear, Softmax+CE**. For each: write `forward`, derive backward on paper, code the backward, `gradcheck` it.
2. Re-derive on paper: **OLS normal equations**, **Ridge closed form**, **the bias term when fitting a model without an intercept** (Citadel's "what happens if true model has intercept but you fit without?" — classic).
3. Write a 30-line tabular pipeline end-to-end from a CSV: load → impute → encode → split → fit → score. Time yourself.

---

### What's next in this series

This notebook covers PyTorch (your stated weakest) plus a pandas appendix. Adjacent notebooks I'd suggest building next:

- **Pandas + EDA + feature engineering deep dive** — the full version of section 9, with the HRT insurance and Jump claims problems worked end-to-end including diagnostics, target encoding, time-aware CV.
- **Classical ML for QR** — OLS / Ridge / Lasso derivations, the multicollinearity questions, the correlated-Y / GLS scenarios from Citadel's final round.
- **Probability & estimation** — KL divergence, Cramer-Rao, the median-of-uniforms problem, the dice/uniform games.

Tell me which to do next.
